# Eddy vertical velocity climatology

Test whether cyclonic eddies (CEs) have positive core-mean vertical velocity and anticyclonic eddies (AEs) negative core-mean vertical velocity in the upper 1,000 m. Model `w` is positive upward. Per-depth centres and core ellipses come from the fitted vertical profiles. The model's extra surface `s_w` level is dropped so its remaining 30 levels align with `z_r`. This notebook is separate from the snapshot explorer and uses a reproducible, polarity-balanced sample by default.


In [ ]:
from pathlib import Path
import json, sys
import netCDF4 as nc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
if HERE.name != 'vertical_velocity':
    raise RuntimeError('Launch from the vertical_velocity folder')
sys.path.insert(0, str(HERE.parent))
sys.path.insert(0, str(HERE.parent/'case_studies'/'eddy_cross_sections_3d'))
import seacofs_tilt_tools as tilt
from case_section_tools import resolve_model_file
from vertical_velocity_tools import sample_snapshot, column_extrema, load_vertical_velocity_xyz


In [ ]:
# Sampling controls. Set N_EDDIES_PER_CLASS=None to use all eligible eddies.
N_EDDIES_PER_CLASS = 150
MAX_DAYS_PER_EDDY = 3
MIN_FITTED_DEPTHS = 8
MAX_DEPTH_M = 1000
FRACTIONS = (0.5, 1.0, 1.5)
MAX_MISMATCH_M = 75
MIN_CORE_CELLS = 8
MIN_COVERAGE = 0.7
SEED = 731
REBUILD = False
MODEL_ROOT = Path('/srv/scratch/z3533156/26year_BRAN2020')
CACHE_ROOT = Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/vertical_velocity_climatology')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_PATH = CACHE_ROOT/'core_vertical_velocity_upper_1000m.parquet'
META_PATH = CACHE_ROOT/'core_vertical_velocity_upper_1000m.json'


In [ ]:
surface, _ = tilt.load_tilt_tables()
vertical = tilt.load_vert()
grid = tilt.load_grid()
profiles = vertical.loc[vertical.Depth.between(0, MAX_DEPTH_M)].copy()
counts = profiles.groupby(['Eddy','Day']).Depth.nunique().rename('n_depths').reset_index()
eligible = surface.loc[surface.Cyc.isin(['AE','CE'])].merge(counts,on=['Eddy','Day'],how='inner')
eligible = eligible.loc[eligible.n_depths.ge(MIN_FITTED_DEPTHS)].copy()
assert not eligible.duplicated(['Eddy','Day']).any()
selection = []
for cyc in ('AE','CE'):
    sub = eligible.loc[eligible.Cyc.eq(cyc)]
    ids = sub.Eddy.drop_duplicates().sample(n=min(N_EDDIES_PER_CLASS,sub.Eddy.nunique()),random_state=SEED) if N_EDDIES_PER_CLASS is not None else sub.Eddy.drop_duplicates()
    for eddy in ids:
        days = sub.loc[sub.Eddy.eq(eddy)]
        selection.append(days.sample(n=min(MAX_DAYS_PER_EDDY,len(days)),random_state=SEED+int(eddy)))
selected = pd.concat(selection,ignore_index=True).sort_values(['fname','Day','Eddy']).reset_index(drop=True)
print(selected.groupby('Cyc').agg(eddies=('Eddy','nunique'),eddy_days=('Day','size')))
display(selected.groupby('Cyc').n_depths.describe())


The sample is chosen using profile availability and polarity, without looking at model vertical velocity or tilt. It is an exploratory climatology, not a perfectly matched AE/CE comparison by latitude, season, region or eddy size. The figures below report those differences for interpretation.


In [ ]:
settings = dict(selected_keys=selected[['Eddy','Day']].astype(int).values.tolist(),
                max_depth_m=MAX_DEPTH_M, fractions=list(FRACTIONS),
                max_mismatch_m=MAX_MISMATCH_M, min_fitted_depths=MIN_FITTED_DEPTHS)
if CACHE_PATH.exists() and not REBUILD:
    if not META_PATH.exists() or json.loads(META_PATH.read_text()) != settings:
        raise ValueError('Saved cache settings differ. Set REBUILD=True to overwrite the single cache.')
    per_depth = pd.read_parquet(CACHE_PATH)
else:
    parts = []
    for fname, rows in selected.groupby('fname',sort=False):
        path = resolve_model_file(rows.iloc[0],MODEL_ROOT)
        print(path.name, len(rows),'eddy-days')
        with nc.Dataset(path) as ds:
            loaded_day = None
            for row in rows.itertuples(index=False):
                if row.Day != loaded_day:
                    velocity_xyz = load_vertical_velocity_xyz(ds,row.Day,grid)
                    loaded_day = row.Day

                fitted = profiles.loc[profiles.Eddy.eq(row.Eddy)&profiles.Day.eq(row.Day)].sort_values('Depth')
                for frac in FRACTIONS:
                    # Geometry must follow the fitted centre at each depth.
                    depth_rows = []
                    for fit in fitted.itertuples(index=False):
                        d,_ = sample_snapshot(ds,fit,[fit.Depth],grid,fractions=(frac,),
                            max_depth_m=MAX_DEPTH_M,max_mismatch_m=MAX_MISMATCH_M,return_maps=False,
                            velocity_xyz=velocity_xyz)
                        if not d.empty: depth_rows.append(d)
                    if depth_rows:
                        part = pd.concat(depth_rows,ignore_index=True)
                        part['Cyc'] = row.Cyc
                        part['TiltDis'] = row.TiltDis
                        parts.append(part)
    if not parts: raise ValueError('No vertical-velocity samples were produced')
    per_depth = pd.concat(parts,ignore_index=True)
    per_depth.to_parquet(CACHE_PATH,index=False)
    META_PATH.write_text(json.dumps(settings,indent=2))
print('Raw sampled rows:',len(per_depth))


In [ ]:
qc = per_depth.loc[per_depth.n_valid.ge(MIN_CORE_CELLS)&per_depth.coverage.ge(MIN_COVERAGE)].copy()
qc = qc.dropna(subset=['w_mean','w_max','w_min'])
print('Retained eddy-days and eddies by polarity:')
display(qc.groupby('Cyc').agg(eddies=('Eddy','nunique'),sampled_days=('Day','size'),rows=('Depth','size'),median_coverage=('coverage','median')))
print('Fraction of per-depth rows retained:',len(qc)/len(per_depth))
column = column_extrema(qc)
identity = selected[['Eddy','Day','Cyc','TiltDis']].drop_duplicates(['Eddy','Day'])
column = column.merge(identity,on=['Eddy','Day'],how='left',validate='many_to_one')


## Depth profiles

For each fitted depth, average days within each eddy first, then compare eddies. The line is the mean of eddy means; the shaded interval is an eddy bootstrap 95% interval. Positive values indicate upwelling. These are pointwise intervals, not a simultaneous test across depths.


In [ ]:
def eddy_bootstrap(frame,value,seed=SEED,n_boot=1000):
    vals = frame.groupby('Eddy')[value].mean().dropna().to_numpy(float)
    if len(vals)<2: return (np.nan,np.nan,np.nan,len(vals))
    rng=np.random.default_rng(seed)
    draws=rng.choice(vals,size=(n_boot,len(vals)),replace=True).mean(axis=1)
    return (vals.mean(),*np.quantile(draws,[.025,.975]),len(vals))

summary=[]
for (frac,depth,cyc),g in qc.groupby(['fraction','Depth','Cyc']):
    for metric in ('w_mean','w_rms','w_max','w_min'):
        est,lo,hi,n=eddy_bootstrap(g,metric)
        summary.append(dict(fraction=frac,Depth=depth,Cyc=cyc,metric=metric,
                            estimate=est,lo=lo,hi=hi,n_eddies=n))
summary=pd.DataFrame(summary)
fig,axes=plt.subplots(1,3,figsize=(16,6),sharey=True)
for ax,frac in zip(axes,FRACTIONS):
    for cyc,color in [('AE','tab:red'),('CE','tab:blue')]:
        g=summary.query('fraction==@frac and Cyc==@cyc and metric=="w_mean"').sort_values('Depth')
        ax.plot(1000*g.estimate,g.Depth,'o-',color=color,label=cyc)
        ax.fill_betweenx(g.Depth,1000*g.lo,1000*g.hi,color=color,alpha=.18)
    ax.axvline(0,color='0.4',lw=.8)
    ax.set(xlabel='Core-mean upward w (mm/s)',title=f'Core fraction {frac:g}')
    ax.legend()
axes[0].set_ylabel('Fitted depth (m)')
axes[0].invert_yaxis()
fig.tight_layout();plt.show()


## AE versus CE contrast

The polarity test uses one mean per eddy at each depth. Its CE-minus-AE interval is obtained by independently resampling eddies within each class. A positive contrast means CEs have more upward motion.


In [ ]:
def contrast_bootstrap(ae,ce,n_boot=1000,seed=SEED):
    a=ae.groupby('Eddy').w_mean.mean().dropna().to_numpy(float)
    c=ce.groupby('Eddy').w_mean.mean().dropna().to_numpy(float)
    if min(len(a),len(c))<2:return (np.nan,np.nan,np.nan,len(a),len(c))
    rng=np.random.default_rng(seed)
    draws=(rng.choice(c,(n_boot,len(c)),replace=True).mean(axis=1)
           -rng.choice(a,(n_boot,len(a)),replace=True).mean(axis=1))
    return (c.mean()-a.mean(),*np.quantile(draws,[.025,.975]),len(a),len(c))
contrasts=[]
for (frac,depth),g in qc.groupby(['fraction','Depth']):
    est,lo,hi,na,nc=contrast_bootstrap(g[g.Cyc.eq('AE')],g[g.Cyc.eq('CE')])
    contrasts.append(dict(fraction=frac,Depth=depth,CE_minus_AE=est,lo=lo,hi=hi,AE_eddies=na,CE_eddies=nc))
contrasts=pd.DataFrame(contrasts)
fig,axes=plt.subplots(1,3,figsize=(16,6),sharey=True)
for ax,frac in zip(axes,FRACTIONS):
    g=contrasts.query('fraction==@frac').sort_values('Depth')
    ax.errorbar(1000*g.CE_minus_AE,g.Depth,xerr=[1000*(g.CE_minus_AE-g.lo),1000*(g.hi-g.CE_minus_AE)],fmt='o')
    ax.axvline(0,color='0.4',lw=.8)
    ax.set(xlabel='CE minus AE core mean (mm/s)',title=f'Core fraction {frac:g}')
axes[0].set_ylabel('Fitted depth (m)');axes[0].invert_yaxis();fig.tight_layout();plt.show()
display(contrasts)


## One summary per eddy-day

The column mean is weighted by the number of valid core cells at each fitted depth. It is a mean over sampled depth-cell values, **not** a physical volume flux: levels and cells have unequal physical spacing and area. The column maximum and minimum are the most positive and negative sampled cells in the upper 1,000 m. Inspect their sensitivity to core fraction before interpreting them.


In [ ]:
metrics=['w_mean','w_rms','w_max','w_min']
eddy_column=column.groupby(['Eddy','Cyc','fraction'],as_index=False)[metrics].mean()
for frac,g in eddy_column.groupby('fraction'):
    print('Core fraction',frac)
    display(g.groupby('Cyc')[metrics].agg(['median','mean','count']))
    for cyc in ('AE','CE'):
        h=g[g.Cyc.eq(cyc)]
        print(cyc,'eddy fraction with upward column mean:',(h.w_mean>0).mean(),'n=',len(h))
fig,axes=plt.subplots(1,len(FRACTIONS),figsize=(15,4),sharey=True)
for ax,frac in zip(axes,FRACTIONS):
    g=eddy_column.query('fraction==@frac')
    ax.boxplot([1000*g[g.Cyc.eq(cyc)].w_mean.dropna() for cyc in ('AE','CE')],labels=['AE','CE'],showfliers=False)
    ax.axhline(0,color='0.4',lw=.8)
    ax.set(title=f'Core fraction {frac:g}',ylabel='Column sample mean w (mm/s)')
fig.tight_layout();plt.show()


In [ ]:
# Audit sampling differences before interpreting a polarity contrast.
audit=selected[['Eddy','Day','Cyc','yc','TiltDis']].drop_duplicates(['Eddy','Day']).copy()
display(audit.groupby('Cyc').agg(eddies=('Eddy','nunique'),days=('Day','size'),
                                  median_y_km=('yc','median'),median_tilt_km=('TiltDis','median')))
display(qc.groupby(['Cyc','fraction']).agg(median_core_cells=('n_valid','median'),
                                            median_coverage=('coverage','median'),
                                            eddies=('Eddy','nunique')))


A positive CE-minus-AE depth profile supports the proposed polarity contrast in the sampled population. It does not by itself establish eddy pumping as the cause; geography, season, background flow and topographic setting can differ between AEs and CEs. The next pass should match or adjust those conditions if a clear contrast appears.
